<a href="https://colab.research.google.com/github/virwang/Fanshawe_DataVisualization26W/blob/main/Text_Summarization_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
text = """
Tom loved to read stories about space. Every night, he looked at the stars and dreamed of visiting them.
One day, he found an old telescope in the attic. He cleaned it and started watching the moon.
Soon, he learned the names of planets and constellations.
Tom decided that one day he would become an astronaut.
He studied hard and never stopped dreaming about the stars.
"""

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

**Frequency-Based Methods (Statistical)**

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize,sent_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import FreqDist
from heapq import nlargest
# import string

#Initizlize Tools
stop_words = set(nltk.corpus.stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Toknize and lemmatize words
words = [lemmatizer.lemmatize(w.lower()) for w in word_tokenize(text) if w.isalpha() and w.lower() not in stop_words]

#Compute frequency distribution
freq_dist = FreqDist(words)

# Store each sentance based in word frequencies
sentences = sent_tokenize(text)
sentence_scores ={}

for sent in sentences:
    for word in word_tokenize(sent.lower()):
        lemma = lemmatizer.lemmatize(word)
        if lemma in freq_dist:
          sentence_scores[sent] = sentence_scores.get(sent, 0) + freq_dist[lemma]
          # print(f"sentence_scores: {sentence_scores}")

# Select top 3 sentences using nlargest
summary_sentences = nlargest(3, sentence_scores, key=sentence_scores.get)

# Preserve  original order of sentences
summary = [sent for sent in sentences if sent in summary_sentences]

# Display the summary
print("\n".join(summary))

Every night, he looked at the stars and dreamed of visiting them.
One day, he found an old telescope in the attic.
Tom decided that one day he would become an astronaut.


**Latent Semantic Analysis (LSA)**

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.tokenize import sent_tokenize
import numpy as np

# Sentence tokenizarion
sentences = sent_tokenize(text)

#TF-IDF representation: Convert each sentence into a TF-IDF vector
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(sentences)

# print(X)

lsa = TruncatedSVD(n_components=1, random_state=42)
lsa.fit(X)

# Compute sentence importance
# Each sentence gets a score based on how much it contributes to the strongest latent
sentence_scores = lsa.transform(X).flatten()

# Select top sentences
# top_sentence_indices = sentence_scores.argsort()[-3:][::-1]

top_sentence_indices = sentence_scores.argsort()[-3:]

print(top_sentence_indices)

#Preserve original sentence order for readability
# summary = [sentences[i] for i in sorted(top_sentence_indices)]

summary = [sentences[i] for i in top_sentence_indices]

print("\n".join(summary))

[0 2 5]

Tom loved to read stories about space.
One day, he found an old telescope in the attic.
Tom decided that one day he would become an astronaut.


**TextRank**

In [ ]:
!pip install networkx
!pip install nltk.metrics

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import networkx as nx


sentences = sent_tokenize(text)

stop_words = set(stopwords.words('english'))

## Lowercase, remove stopwords and non-alphabetic tokens.
def preprocess(sentence):
  words = word_tokenize(sentence.lower())
  return " ".join([w for w in words if w.isalpha() and w not in stop_words])

clean_sentences = [preprocess(s) for s in sentences]

#Convert sentences to TF-IDF vectors
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(clean_sentences)

#Compute cosine simiarity between each pari of sentence
similarity_matrix = cosine_similarity(X, X)

# Build a simiarity graph: Each sentence is a node, edges represent similarity scores
nx_graph = nx.from_numpy_array(similarity_matrix)

#Apply PageRank algorithm: Rank sentences by theur overall importance in the network.
scores = nx.pagerank(nx_graph)


# Rank sentences by score (highest first): [(score_1, sentence_1), (score_2, sentence_2), ..., (score_n, sentence_n)]
ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)

# Select top N sentences
top_n = 3
summary_sentences = [s for _, s in ranked_sentences[:top_n]]

# Preserve original order for readability
summary = [s for s in sentences if s in summary_sentences]

# Display the summary
print("\n".join(summary))

Soon, he learned the names of planets and constellations.
Tom decided that one day he would become an astronaut.
He studied hard and never stopped dreaming about the stars.
